In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_parquet("../data/video_index.parquet")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (217, 388)


,video_id,title,datetime,transcript,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,KGVpKPNUdzA,Khabib vs Lex: Training with Khabib | FULL EXC...,2026-02-25,[Lex] All right. We're training with the Khabi...,-0.011134,0.029237,0.023949,-0.052703,-0.037568,0.071175,...,0.090085,0.050960,-0.084281,0.021811,-0.037674,-0.013464,-0.025635,0.015907,-0.008922,0.011621
1,YFjfBk8HI5o,OpenClaw: The Viral AI Agent that Broke the In...,2026-02-12,"- I watched my agent happily click the ""I'm no...",-0.006191,-0.107095,-0.031676,-0.046824,0.022702,0.033652,...,0.015205,0.018775,0.020557,-0.064464,-0.017788,0.058181,0.001029,0.017298,0.048867,0.009677
2,EV7WhVT270Q,"State of AI in 2026: LLMs, Coding, Scaling Law...",2026-01-31,- The following is a conversation all about th...,-0.047564,-0.113781,-0.008190,-0.058883,0.032966,0.123650,...,0.076698,0.053705,-0.061308,-0.066854,-0.065806,0.069400,0.043714,-0.037885,-0.013278,0.078460
3,Z-FRe5AKmCU,Paul Rosolie: Uncontacted Tribes in the Amazon...,2026-01-13,- ... were standing there. Everyone is waiting...,0.005778,-0.041444,0.027730,0.029519,0.096083,-0.015146,...,0.085844,0.018410,0.005107,0.060228,-0.018694,0.055271,0.010633,-0.007049,-0.069700,0.001311
4,14OPT6CcsH4,"Infinity, Paradoxes, Gödel Incompleteness &amp...",2025-12-31,- The following is a conversation with Joel Da...,-0.058031,-0.040032,0.004658,0.048868,0.000742,-0.019905,...,0.063012,-0.004456,-0.059840,0.023605,-0.202434,0.035830,0.110208,0.006893,-0.049519,-0.012066


In [2]:
embedding_cols = [col for col in df.columns if col.startswith("emb_")]
video_embeddings = df[embedding_cols].values
print("Embedding matrix shape:", video_embeddings.shape)

Embedding matrix shape: (217, 384)


In [3]:
BEST_MODEL = "multi-qa-MiniLM-L6-cos-v1"
model = SentenceTransformer(BEST_MODEL)
print("Model loaded successfully")

Model loaded successfully


In [4]:
def returnSearchResults(query, df, top_k=5, threshold=0.7):

    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, video_embeddings)[0]
    threshold = similarities.max() * 0.7
    sorted_indices = np.argsort(similarities)[::-1]

    results = []

    for idx in sorted_indices[:top_k]:

        score = similarities[idx]

        if score >= threshold:
            results.append({
                "video_id": df.iloc[idx]["video_id"],
                "title": df.iloc[idx]["title"],
                "score": float(score)
            })

    if len(results) == 0:
        for idx in sorted_indices[:top_k]:
            results.append({
                "video_id": df.iloc[idx]["video_id"],
                "title": df.iloc[idx]["title"],
                "score": float(similarities[idx])
            })

    return pd.DataFrame(results)

query = "What is reinforcement learning?"
results = returnSearchResults(query, df)
print(results.head())

      video_id                                              title     score
0  kxi-_TT_-Nc  Sergey Levine: Robotics and Machine Learning |...  0.408473
1  HhY95m-WD_E  Dawn Song: Adversarial Machine Learning and Co...  0.354578
2  e77NkSjnyH4  AlphaZero and Self Play (David Silver, DeepMin...  0.344421
3  tNZnLkRBYA8  ThePrimeagen: Programming, AI, ADHD, Productiv...  0.332504
4  3wMKoSRbGVs  Douglas Lenat: Cyc and the Quest to Solve Comm...  0.301739
